# Retail market-basket analysis

Prepare megastore orders and identify product associations using Apriori. Support, confidence and lift describe observed co-purchases that could inform merchandising experiments; they do not establish that a promotion will increase sales.


## Load and inspect orders


In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv("../data/megastore_transactions.csv")

print(df.head())

   OrderID                         ProductName  Quantity     InvoiceDate  \
0   536370         INFLATABLE POLITICAL GLOBE         48  12/1/2010 8:45   
1   536370      SET2 RED RETROSPOT TEA TOWELS         18  12/1/2010 8:45   
2   536370     PANDA AND BUNNIES STICKER SHEET        12  12/1/2010 8:45   
3   536370       RED TOADSTOOL LED NIGHT LIGHT        24  12/1/2010 8:45   
4   536370  VINTAGE HEADS AND TAILS CARD GAME         24  12/1/2010 8:45   

  UnitPrice  TotalCost         Country DiscountApplied OrderPriority  \
0     $0.85     $40.80   United States             Yes          High   
1     $2.95     $53.10   United States             Yes          High   
2     $0.85     $10.20   United States             Yes          High   
3     $1.65     $39.60   United States             Yes          High   
4     $1.25     $30.00   United States             Yes          High   

      Region    Segment ExpeditedShipping PaymentMethod  \
0  Northeast  Corporate               Yes   Credit 

In [2]:
print(df.columns)

Index(['OrderID', 'ProductName', 'Quantity', 'InvoiceDate', 'UnitPrice ',
       'TotalCost ', 'Country', 'DiscountApplied', 'OrderPriority', 'Region',
       'Segment', 'ExpeditedShipping', 'PaymentMethod',
       'CustomerOrderSatisfaction'],
      dtype='object')


In [3]:
print(df.dtypes)
print(df.shape)

OrderID                       int64
ProductName                  object
Quantity                      int64
InvoiceDate                  object
UnitPrice                    object
TotalCost                    object
Country                      object
DiscountApplied              object
OrderPriority                object
Region                       object
Segment                      object
ExpeditedShipping            object
PaymentMethod                object
CustomerOrderSatisfaction    object
dtype: object
(8234, 14)


In [4]:
print(df.nunique())

OrderID                       441
ProductName                  1562
Quantity                       57
InvoiceDate                   441
UnitPrice                      86
TotalCost                     443
Country                         1
DiscountApplied                 2
OrderPriority                   2
Region                          2
Segment                         2
ExpeditedShipping               2
PaymentMethod                   2
CustomerOrderSatisfaction       5
dtype: int64


## Select and clean transaction fields


In [5]:
columns_of_interest = ['Country', 'DiscountApplied', 'OrderPriority', 'Region', 'Segment', 'ExpeditedShipping', 'PaymentMethod', 'CustomerOrderSatisfaction']

for column in columns_of_interest:
    print(f"Unique values for {column}:")
    print(df[column].unique())
    print()

Unique values for Country:
['United States']

Unique values for DiscountApplied:
['Yes' 'No']

Unique values for OrderPriority:
['High' 'Medium']

Unique values for Region:
['Northeast' 'Southeast']

Unique values for Segment:
['Corporate' 'Consumer']

Unique values for ExpeditedShipping:
['Yes' 'No']

Unique values for PaymentMethod:
['Credit Card' 'PayPal']

Unique values for CustomerOrderSatisfaction:
['Satisfied' 'Dissatisfied' 'Prefer not to answer' 'Very Satisfied'
 'Very Dissatisfied']



In [6]:
print(df.isnull().sum())

OrderID                      0
ProductName                  0
Quantity                     0
InvoiceDate                  0
UnitPrice                    0
TotalCost                    0
Country                      0
DiscountApplied              0
OrderPriority                0
Region                       0
Segment                      0
ExpeditedShipping            0
PaymentMethod                0
CustomerOrderSatisfaction    0
dtype: int64


In [7]:
df.columns = df.columns.str.strip()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'].replace(r'[\$,]', '', regex=True), errors='coerce')
df['TotalCost'] = pd.to_numeric(df['TotalCost'].replace(r'[\$,]', '', regex=True), errors='coerce')

print(df.dtypes)


OrderID                               int64
ProductName                          object
Quantity                              int64
InvoiceDate                  datetime64[ns]
UnitPrice                           float64
TotalCost                           float64
Country                              object
DiscountApplied                      object
OrderPriority                        object
Region                               object
Segment                              object
ExpeditedShipping                    object
PaymentMethod                        object
CustomerOrderSatisfaction            object
dtype: object


In [8]:
print(df.isnull().sum())

OrderID                      0
ProductName                  0
Quantity                     0
InvoiceDate                  0
UnitPrice                    0
TotalCost                    0
Country                      0
DiscountApplied              0
OrderPriority                0
Region                       0
Segment                      0
ExpeditedShipping            0
PaymentMethod                0
CustomerOrderSatisfaction    0
dtype: int64


In [9]:
print(df.describe())

             OrderID     Quantity                    InvoiceDate    UnitPrice  \
count    8234.000000  8234.000000                           8234  8234.000000   
mean   560874.506923    13.705125  2011-07-13 07:41:26.035948288     2.675130   
min    536370.000000     1.000000            2010-12-01 08:45:00     0.060000   
25%    549274.000000     6.000000            2011-04-07 13:07:00     1.060000   
50%    563502.000000    10.000000            2011-08-17 08:50:00     1.650000   
75%    571864.000000    12.000000            2011-10-19 13:49:00     2.950000   
max    581587.000000   912.000000            2011-12-09 12:50:00    42.950000   
std     13082.500625    21.494536                            NaN     2.770026   

         TotalCost  
count  8234.000000  
mean     22.710966  
min       0.290000  
25%      10.500000  
50%      16.500000  
75%      22.500000  
max    1632.480000  
std      34.893061  


In [10]:
print(df.duplicated().sum())

16


In [11]:
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

0


## Encode orders and build baskets


In [12]:
from IPython.display import display, HTML

example = df.head(5)

display(HTML(example.to_html(index=False)))


OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,DiscountApplied,OrderPriority,Region,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction
536370,INFLATABLE POLITICAL GLOBE,48,2010-12-01 08:45:00,0.85,40.8,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
536370,SET2 RED RETROSPOT TEA TOWELS,18,2010-12-01 08:45:00,2.95,53.1,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
536370,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,10.2,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
536370,RED TOADSTOOL LED NIGHT LIGHT,24,2010-12-01 08:45:00,1.65,39.6,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
536370,VINTAGE HEADS AND TAILS CARD GAME,24,2010-12-01 08:45:00,1.25,30.0,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied


In [13]:
from IPython.display import display, HTML

# Preserve the order of priority and satisfaction categories.
priority_mapping = {'High': 3, 'Medium': 2, 'Low': 1}
satisfaction_mapping = {
    'Very Satisfied': 4,
    'Satisfied': 3,
    'Dissatisfied': 2,
    'Very Dissatisfied': 1
}

df['OrderPriority'] = df['OrderPriority'].map(priority_mapping).astype('Int64')
df['CustomerOrderSatisfaction'] = df['CustomerOrderSatisfaction'].map(satisfaction_mapping).astype('Int64')

# Encode region and discount flags without imposing a category order.
df_nominal = pd.get_dummies(df[['Region', 'DiscountApplied']], drop_first=False)

df_encoded = pd.concat([df, df_nominal], axis=1)
df_encoded = df_encoded.drop(['Region', 'DiscountApplied'], axis=1)

display(HTML(df_encoded.head().to_html(index=False)))

df_encoded.to_excel('encoded_dataset.xlsx', index=False)



OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,OrderPriority,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction,Region_Northeast,Region_Southeast,DiscountApplied_No,DiscountApplied_Yes
536370,INFLATABLE POLITICAL GLOBE,48,2010-12-01 08:45:00,0.85,40.8,United States,3,Corporate,Yes,Credit Card,3,True,False,False,True
536370,SET2 RED RETROSPOT TEA TOWELS,18,2010-12-01 08:45:00,2.95,53.1,United States,3,Corporate,Yes,Credit Card,3,True,False,False,True
536370,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,10.2,United States,3,Corporate,Yes,Credit Card,3,True,False,False,True
536370,RED TOADSTOOL LED NIGHT LIGHT,24,2010-12-01 08:45:00,1.65,39.6,United States,3,Corporate,Yes,Credit Card,3,True,False,False,True
536370,VINTAGE HEADS AND TAILS CARD GAME,24,2010-12-01 08:45:00,1.25,30.0,United States,3,Corporate,Yes,Credit Card,3,True,False,False,True


In [14]:
from mlxtend.preprocessing import TransactionEncoder

# Each OrderID defines one basket of products.
basket = df.groupby('OrderID')['ProductName'].apply(list).tolist()

te = TransactionEncoder()
te_ary = te.fit(basket).transform(basket)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

display(df_basket.head())

df_basket.to_excel('basket_encoded.xlsx', index=False)

,50S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TRELLIS COAT RACK,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,YELLOW SHARK HELICOPTER,ZINC STAR TLIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC TLIGHT HOLDER STAR LARGE,ZINC TLIGHT HOLDER STARS SMALL
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Mine frequent itemsets and association rules


In [15]:
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display, HTML

# Retain itemsets present in at least 1% of baskets.
frequent_itemsets = apriori(df_basket, min_support=0.01, use_colnames=True)

# Require lift of at least 1 before ranking the rules.
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

display(HTML(rules.head().to_html(index=False)))

antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
(CHARLOTTE BAG DOLLY GIRL DESIGN),( DOLLY GIRL BEAKER),0.058957,0.020408,0.011338,0.192308,9.423077,1.0,0.010135,1.212828,0.949880,0.166667,0.175481,0.373932
( DOLLY GIRL BEAKER),(CHARLOTTE BAG DOLLY GIRL DESIGN),0.020408,0.058957,0.011338,0.555556,9.423077,1.0,0.010135,2.117347,0.912500,0.166667,0.527711,0.373932
(DOLLY GIRL CHILDRENS BOWL),( DOLLY GIRL BEAKER),0.040816,0.020408,0.015873,0.388889,19.055556,1.0,0.015040,1.602968,0.987842,0.350000,0.376157,0.583333
( DOLLY GIRL BEAKER),(DOLLY GIRL CHILDRENS BOWL),0.020408,0.040816,0.015873,0.777778,19.055556,1.0,0.015040,4.316327,0.967262,0.350000,0.768322,0.583333
(DOLLY GIRL CHILDRENS CUP),( DOLLY GIRL BEAKER),0.036281,0.020408,0.013605,0.375000,18.375000,1.0,0.012865,1.567347,0.981176,0.315789,0.361979,0.520833


## Interpret confidence and lift


In [16]:
# Read confidence and lift alongside support.

# Confidence is P(consequent | antecedent); lift compares it with P(consequent).

rules_sorted_lift = rules.sort_values(by='lift', ascending=False)

pd.set_option('display.max_colwidth', None)

rules_sorted_lift.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
82991,"(CARD DOLLY GIRL , CHILDRENS CUTLERY SPACEBOY , ALARM CLOCK BAKELIKE PINK)","(SPACEBOY BIRTHDAY CARD, ROUND SNACK BOXES SET OF4 WOODLAND , CHILDRENS CUTLERY DOLLY GIRL , ALARM CLOCK BAKELIKE RED )",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
83013,"(CARD DOLLY GIRL , ALARM CLOCK BAKELIKE PINK)","(SPACEBOY BIRTHDAY CARD, ROUND SNACK BOXES SET OF4 WOODLAND , CHILDRENS CUTLERY SPACEBOY , ALARM CLOCK BAKELIKE RED , CHILDRENS CUTLERY DOLLY GIRL )",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
75207,"(SPACEBOY BIRTHDAY CARD, CHILDRENS CUTLERY SPACEBOY , ALARM CLOCK BAKELIKE PINK)","(ROUND SNACK BOXES SET OF4 WOODLAND , CARD DOLLY GIRL , ALARM CLOCK BAKELIKE RED )",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
75206,"(SPACEBOY BIRTHDAY CARD, CHILDRENS CUTLERY SPACEBOY , ALARM CLOCK BAKELIKE RED )","(ROUND SNACK BOXES SET OF4 WOODLAND , CARD DOLLY GIRL , ALARM CLOCK BAKELIKE PINK)",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
82113,"(SET6 RED SPOTTY PAPER PLATES, ROUND SNACK BOXES SET OF4 WOODLAND , ALARM CLOCK BAKELIKE PINK)","(PLASTERS IN TIN CIRCUS PARADE , SET6 RED SPOTTY PAPER CUPS, ALARM CLOCK BAKELIKE GREEN, ALARM CLOCK BAKELIKE RED )",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
